# IESO Coincident Peak Prediction — Model Training & Selection

This notebook trains and compares multiple approaches for predicting daily maximum
Ontario demand:
- **Approach A:** Daily max demand regression (Linear, Random Forest, XGBoost)
- **Approach B:** Peak day classification (handles class imbalance)
- **Approach C:** Threshold-based heuristic (baseline)

The regression approach is expected to outperform classification because it avoids
the extreme class imbalance problem (5 peaks out of 8,760 hours = 0.057%).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
from pathlib import Path
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                             precision_score, recall_score, f1_score,
                             confusion_matrix, precision_recall_curve,
                             classification_report)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import lightgbm as lgb
import shap

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)

PROJECT_ROOT = Path(r'C:/wamp64/www/Spec_Driven_Dev_Website')
DATA_DIR = PROJECT_ROOT / 'notebooks' / 'source' / 'data'
MODEL_DIR = PROJECT_ROOT / 'notebooks' / 'source' / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print('Libraries loaded successfully')

In [ ]:
# Load feature-engineered dataset
features = pd.read_parquet(DATA_DIR / 'ieso_features_daily.parquet')
features['Date'] = pd.to_datetime(features['Date'])

print(f'Feature matrix: {len(features)} days x {len(features.columns)} columns')
print(f'Date range: {features["Date"].min()} to {features["Date"].max()}')
print(f'Base periods: {sorted(features["base_period"].unique())}')
print(f'Peak days: {features["is_peak_day"].sum()}')

## Train / Validation / Test Split

Split by base period boundary to prevent temporal leakage:
- **Train:** Base periods 2010–2021 (12 years)
- **Validation:** Base periods 2022–2023 (2 years)
- **Test:** Base period 2024 (held out, never seen during development)

In [ ]:
# Define feature columns for modeling
FEATURE_COLS = [
    # Weather features (primary)
    'daily_max_temp', 'daily_mean_temp', 'daily_max_humidex', 'daily_cdh',
    'daily_mean_rh', 'daily_mean_dewpoint', 'temp_3day_avg', 'cdh_3day_avg',
    # Temporal features
    'month', 'day_of_week', 'is_business_day', 'day_of_year',
    # Demand momentum (available by morning)
    'prev_day_max_demand', 'rolling_7d_max_demand', 'rolling_7d_mean_demand',
    # Peak context
    'current_5th_peak', 'max_demand_so_far',
]

TARGET_COL = 'daily_max_demand'
PEAK_COL = 'is_peak_day'

# Split by base period
train_mask = features['base_period'].isin(range(2010, 2022))
val_mask = features['base_period'].isin([2022, 2023])
test_mask = features['base_period'] == 2024

# Drop rows with missing features
complete_mask = features[FEATURE_COLS + [TARGET_COL]].notna().all(axis=1)

train = features[train_mask & complete_mask].copy()
val = features[val_mask & complete_mask].copy()
test = features[test_mask & complete_mask].copy()

X_train, y_train = train[FEATURE_COLS], train[TARGET_COL]
X_val, y_val = val[FEATURE_COLS], val[TARGET_COL]
X_test, y_test = test[FEATURE_COLS], test[TARGET_COL]

# Classification targets
y_train_cls = train[PEAK_COL]
y_val_cls = val[PEAK_COL]
y_test_cls = test[PEAK_COL]

print(f'Train: {len(train)} days (base periods 2010–2021), '
      f'{y_train_cls.sum()} peak days ({y_train_cls.mean()*100:.2f}%)')
print(f'Val:   {len(val)} days (base periods 2022–2023), '
      f'{y_val_cls.sum()} peak days ({y_val_cls.mean()*100:.2f}%)')
print(f'Test:  {len(test)} days (base period 2024), '
      f'{y_test_cls.sum()} peak days ({y_test_cls.mean()*100:.2f}%)')

## Approach A: Daily Max Demand Regression

Predict daily maximum Ontario Demand (MW) as a continuous quantity.
Then derive peak day alerts by comparing predictions to the displacement threshold.

In [ ]:
# Model A1: Linear Regression (baseline)
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr_val = lr.predict(X_val)
y_pred_lr_test = lr.predict(X_test)

print('=== Linear Regression ===')
print(f'Train R²:  {lr.score(X_train, y_train):.4f}')
print(f'Val RMSE:  {np.sqrt(mean_squared_error(y_val, y_pred_lr_val)):.1f} MW')
print(f'Val MAE:   {mean_absolute_error(y_val, y_pred_lr_val):.1f} MW')
print(f'Val R²:    {r2_score(y_val, y_pred_lr_val):.4f}')
print(f'Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_lr_test)):.1f} MW')
print(f'Test R²:   {r2_score(y_test, y_pred_lr_test):.4f}')

In [ ]:
# Model A2: Random Forest Regressor
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf_val = rf.predict(X_val)
y_pred_rf_test = rf.predict(X_test)

print('=== Random Forest ===')
print(f'Train R²:  {rf.score(X_train, y_train):.4f}')
print(f'Val RMSE:  {np.sqrt(mean_squared_error(y_val, y_pred_rf_val)):.1f} MW')
print(f'Val MAE:   {mean_absolute_error(y_val, y_pred_rf_val):.1f} MW')
print(f'Val R²:    {r2_score(y_val, y_pred_rf_val):.4f}')
print(f'Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_rf_test)):.1f} MW')
print(f'Test R²:   {r2_score(y_test, y_pred_rf_test):.4f}')

In [ ]:
# Model A3: XGBoost Regressor
xgb_model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    random_state=42,
    verbosity=0
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

y_pred_xgb_val = xgb_model.predict(X_val)
y_pred_xgb_test = xgb_model.predict(X_test)

print('=== XGBoost ===')
print(f'Train R²:  {xgb_model.score(X_train, y_train):.4f}')
print(f'Val RMSE:  {np.sqrt(mean_squared_error(y_val, y_pred_xgb_val)):.1f} MW')
print(f'Val MAE:   {mean_absolute_error(y_val, y_pred_xgb_val):.1f} MW')
print(f'Val R²:    {r2_score(y_val, y_pred_xgb_val):.4f}')
print(f'Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_xgb_test)):.1f} MW')
print(f'Test R²:   {r2_score(y_test, y_pred_xgb_test):.4f}')

In [ ]:
# Model A4: LightGBM Regressor
lgb_model = lgb.LGBMRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=10,
    random_state=42,
    verbose=-1
)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
)

y_pred_lgb_val = lgb_model.predict(X_val)
y_pred_lgb_test = lgb_model.predict(X_test)

print('=== LightGBM ===')
print(f'Train R²:  {lgb_model.score(X_train, y_train):.4f}')
print(f'Val RMSE:  {np.sqrt(mean_squared_error(y_val, y_pred_lgb_val)):.1f} MW')
print(f'Val MAE:   {mean_absolute_error(y_val, y_pred_lgb_val):.1f} MW')
print(f'Val R²:    {r2_score(y_val, y_pred_lgb_val):.4f}')
print(f'Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_lgb_test)):.1f} MW')
print(f'Test R²:   {r2_score(y_test, y_pred_lgb_test):.4f}')

## Approach B: Peak Day Classification

Binary classification with class imbalance handling. Included for comparison
to demonstrate why regression is preferred.

In [ ]:
# Model B1: Logistic Regression with class weights
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

log_clf = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
log_clf.fit(X_train_scaled, y_train_cls)

y_pred_logcls_val = log_clf.predict(X_val_scaled)
y_pred_logcls_test = log_clf.predict(X_test_scaled)

print('=== Logistic Regression (Classification) ===')
print(f'Val  — Precision: {precision_score(y_val_cls, y_pred_logcls_val, zero_division=0):.3f}, '
      f'Recall: {recall_score(y_val_cls, y_pred_logcls_val, zero_division=0):.3f}, '
      f'F1: {f1_score(y_val_cls, y_pred_logcls_val, zero_division=0):.3f}')
print(f'Test — Precision: {precision_score(y_test_cls, y_pred_logcls_test, zero_division=0):.3f}, '
      f'Recall: {recall_score(y_test_cls, y_pred_logcls_test, zero_division=0):.3f}, '
      f'F1: {f1_score(y_test_cls, y_pred_logcls_test, zero_division=0):.3f}')
print(f'Val false alarms: {((y_pred_logcls_val == 1) & (y_val_cls == 0)).sum()}')

In [ ]:
# Model B2: XGBoost Classifier
xgb_clf = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=len(y_train_cls[y_train_cls == 0]) / max(len(y_train_cls[y_train_cls == 1]), 1),
    random_state=42,
    verbosity=0
)
xgb_clf.fit(X_train, y_train_cls, eval_set=[(X_val, y_val_cls)], verbose=False)

y_pred_xgbcls_val = xgb_clf.predict(X_val)
y_pred_xgbcls_test = xgb_clf.predict(X_test)

print('=== XGBoost Classifier ===')
print(f'Val  — Precision: {precision_score(y_val_cls, y_pred_xgbcls_val, zero_division=0):.3f}, '
      f'Recall: {recall_score(y_val_cls, y_pred_xgbcls_val, zero_division=0):.3f}, '
      f'F1: {f1_score(y_val_cls, y_pred_xgbcls_val, zero_division=0):.3f}')
print(f'Test — Precision: {precision_score(y_test_cls, y_pred_xgbcls_test, zero_division=0):.3f}, '
      f'Recall: {recall_score(y_test_cls, y_pred_xgbcls_test, zero_division=0):.3f}, '
      f'F1: {f1_score(y_test_cls, y_pred_xgbcls_test, zero_division=0):.3f}')
print(f'Val false alarms: {((y_pred_xgbcls_val == 1) & (y_val_cls == 0)).sum()}')

## Approach C: Threshold-Based Heuristic (Baseline)

Simple rule: alert if forecast max temp > 30°C AND is_weekday AND month in {6,7,8}.
This represents what an experienced energy manager might do without a model.

In [ ]:
def heuristic_alert(df):
    """Simple temperature heuristic for peak day prediction."""
    return (
        (df['daily_max_temp'] > 30) & 
        (df['is_business_day'] == 1) & 
        (df['month'].isin([6, 7, 8]))
    ).astype(int)

y_pred_heuristic_val = heuristic_alert(val)
y_pred_heuristic_test = heuristic_alert(test)

print('=== Temperature Heuristic (>30°C, weekday, Jun-Aug) ===')
print(f'Val  — Precision: {precision_score(y_val_cls, y_pred_heuristic_val, zero_division=0):.3f}, '
      f'Recall: {recall_score(y_val_cls, y_pred_heuristic_val, zero_division=0):.3f}, '
      f'F1: {f1_score(y_val_cls, y_pred_heuristic_val, zero_division=0):.3f}')
print(f'Test — Precision: {precision_score(y_test_cls, y_pred_heuristic_test, zero_division=0):.3f}, '
      f'Recall: {recall_score(y_test_cls, y_pred_heuristic_test, zero_division=0):.3f}, '
      f'F1: {f1_score(y_test_cls, y_pred_heuristic_test, zero_division=0):.3f}')
print(f'Val alert days: {y_pred_heuristic_val.sum()}')
print(f'Test alert days: {y_pred_heuristic_test.sum()}')

## Regression → Alert Conversion & Model Comparison

Convert regression predictions into RED/YELLOW/GREEN alerts using the
displacement threshold, then evaluate peak detection performance.

In [ ]:
def regression_to_alerts(y_pred, threshold_series, buffer_mw=500):
    """Convert regression predictions to RED/YELLOW/GREEN alerts.
    
    RED:    predicted > threshold + buffer
    YELLOW: |predicted - threshold| <= buffer
    GREEN:  predicted < threshold - buffer
    """
    alerts = pd.Series('GREEN', index=range(len(y_pred)))
    for i in range(len(y_pred)):
        threshold = threshold_series.iloc[i] if threshold_series.iloc[i] > 0 else 20000
        diff = y_pred[i] - threshold
        if diff > buffer_mw:
            alerts.iloc[i] = 'RED'
        elif abs(diff) <= buffer_mw:
            alerts.iloc[i] = 'YELLOW'
    return alerts

# Generate alerts from the best regression model (XGBoost)
val_alerts = regression_to_alerts(y_pred_xgb_val, val['current_5th_peak'].reset_index(drop=True))
test_alerts = regression_to_alerts(y_pred_xgb_test, test['current_5th_peak'].reset_index(drop=True))

# Convert RED alerts to binary for comparison
val_red = (val_alerts == 'RED').astype(int) | (val_alerts == 'YELLOW').astype(int)
test_red = (test_alerts == 'RED').astype(int) | (test_alerts == 'YELLOW').astype(int)

print('=== XGBoost Regression → Alert System ===')
print(f'Val  — Precision: {precision_score(y_val_cls.values, val_red.values, zero_division=0):.3f}, '
      f'Recall: {recall_score(y_val_cls.values, val_red.values, zero_division=0):.3f}')
print(f'Test — Precision: {precision_score(y_test_cls.values, test_red.values, zero_division=0):.3f}, '
      f'Recall: {recall_score(y_test_cls.values, test_red.values, zero_division=0):.3f}')
print(f'\nAlert distribution (test set):')
print(test_alerts.value_counts().to_string())

In [ ]:
# Model Comparison Table
def eval_regression(name, y_true, y_pred, y_true_cls, threshold_series):
    alerts = regression_to_alerts(y_pred, threshold_series)
    alert_binary = ((alerts == 'RED') | (alerts == 'YELLOW')).astype(int)
    return {
        'Model': name,
        'RMSE (MW)': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE (MW)': mean_absolute_error(y_true, y_pred),
        'R²': r2_score(y_true, y_pred),
        'Peak Precision': precision_score(y_true_cls.values, alert_binary.values, zero_division=0),
        'Peak Recall': recall_score(y_true_cls.values, alert_binary.values, zero_division=0),
        'Peak F1': f1_score(y_true_cls.values, alert_binary.values, zero_division=0),
        'Alert Days': alert_binary.sum(),
    }

def eval_classifier(name, y_true_cls, y_pred_cls):
    return {
        'Model': name,
        'RMSE (MW)': np.nan,
        'MAE (MW)': np.nan,
        'R²': np.nan,
        'Peak Precision': precision_score(y_true_cls, y_pred_cls, zero_division=0),
        'Peak Recall': recall_score(y_true_cls, y_pred_cls, zero_division=0),
        'Peak F1': f1_score(y_true_cls, y_pred_cls, zero_division=0),
        'Alert Days': y_pred_cls.sum(),
    }

threshold_val = val['current_5th_peak'].reset_index(drop=True)
threshold_test = test['current_5th_peak'].reset_index(drop=True)

comparison = pd.DataFrame([
    eval_regression('Linear Regression', y_test, y_pred_lr_test, y_test_cls, threshold_test),
    eval_regression('Random Forest', y_test, y_pred_rf_test, y_test_cls, threshold_test),
    eval_regression('XGBoost', y_test, y_pred_xgb_test, y_test_cls, threshold_test),
    eval_regression('LightGBM', y_test, y_pred_lgb_test, y_test_cls, threshold_test),
    eval_classifier('Logistic (balanced)', y_test_cls, y_pred_logcls_test),
    eval_classifier('XGBoost Classifier', y_test_cls, y_pred_xgbcls_test),
    eval_classifier('Heuristic (>30°C)', y_test_cls, y_pred_heuristic_test),
])

print('=== Model Comparison (Test Set: 2024 Base Period) ===')
print(comparison.round(3).to_string(index=False))

## SHAP Feature Importance Analysis

SHAP (SHapley Additive exPlanations) values quantify each feature's contribution
to individual predictions. This reveals what drives the model's decisions.

In [ ]:
# SHAP analysis on the best model (XGBoost)
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_val)

# Summary plot (beeswarm)
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_val, feature_names=FEATURE_COLS, 
                  show=False, max_display=15)
plt.title('SHAP Feature Importance — XGBoost Demand Regression')
plt.tight_layout()
plt.savefig(DATA_DIR / 'shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP dependence plots for top 3 features
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

top_features = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': np.abs(shap_values).mean(axis=0)
}).sort_values('importance', ascending=False).head(3)['feature'].values

for i, feat in enumerate(top_features):
    feat_idx = FEATURE_COLS.index(feat)
    axes[i].scatter(X_val[feat].values, shap_values[:, feat_idx], 
                    alpha=0.3, s=5, color='#1565C0')
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('SHAP Value (MW)')
    axes[i].set_title(f'SHAP Dependence: {feat}')
    axes[i].axhline(y=0, color='gray', linestyle='--', alpha=0.5)

plt.suptitle('SHAP Dependence Plots — Top 3 Features', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(DATA_DIR / 'shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()

## Hyperparameter Tuning

In [ ]:
# Light hyperparameter search on XGBoost
param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.03, 0.05, 0.1],
    'n_estimators': [200, 300, 500],
    'min_child_weight': [3, 5, 10],
}

# Use time-series cross-validation on training data
tscv = TimeSeriesSplit(n_splits=3)
grid_search = GridSearchCV(
    xgb.XGBRegressor(subsample=0.8, colsample_bytree=0.8, 
                     random_state=42, verbosity=0),
    param_grid,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=0
)
grid_search.fit(X_train, y_train)

print(f'Best parameters: {grid_search.best_params_}')
print(f'Best CV RMSE: {-grid_search.best_score_:.1f} MW')

# Retrain with best parameters
best_model = grid_search.best_estimator_
y_pred_best_test = best_model.predict(X_test)
print(f'\nTuned model test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_best_test)):.1f} MW')
print(f'Tuned model test R²: {r2_score(y_test, y_pred_best_test):.4f}')

## Save Trained Model

In [ ]:
# Save the best model and metadata
model_artifact = {
    'model': best_model,
    'feature_columns': FEATURE_COLS,
    'target_column': TARGET_COL,
    'scaler': scaler,  # For classification comparison
    'training_base_periods': list(range(2010, 2022)),
    'validation_base_periods': [2022, 2023],
    'test_base_period': 2024,
    'best_params': grid_search.best_params_,
    'test_rmse': np.sqrt(mean_squared_error(y_test, y_pred_best_test)),
    'test_r2': r2_score(y_test, y_pred_best_test),
}

joblib.dump(model_artifact, MODEL_DIR / 'ieso_peak_model.joblib')
print(f'Model saved to: {MODEL_DIR / "ieso_peak_model.joblib"}')
print(f'Best test RMSE: {model_artifact["test_rmse"]:.1f} MW')
print(f'Best test R²: {model_artifact["test_r2"]:.4f}')

# Also save the comparison table
comparison.to_csv(DATA_DIR / 'model_comparison.csv', index=False)
print(f'\nModel comparison saved to: {DATA_DIR / "model_comparison.csv"}')

print('\n=== Notebook 3 complete ===')